In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.metrics import confusion_matrix
import pandas as pd
import numpy as np
import random
from sklearn.metrics import (
    accuracy_score, recall_score, precision_score,
    f1_score, matthews_corrcoef, roc_auc_score, cohen_kappa_score
)

# SETTINGS
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 512
LR = 2e-3
ROUNDS = 25
LOCAL_EPOCHS = 1
NUM_CLIENTS = 5
FEATURE_COL_COUNT = 128

TRAIN_CSV = "/content/fasttext-train.csv"
TEST_CSV = "/content/fasttext-testM.csv"

# Contrastive learning should use ONLY training data
CONTRASTIVE_CSV = TRAIN_CSV

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

LAMBDA = 0.3  # weight for classification loss

# DATASET
class FeatureDataset(Dataset):
    def __init__(self, csv_file, target_col="Target"):
        df = pd.read_csv(csv_file)
        feature_cols = [c for c in df.columns if c != target_col]

        self.X = df[feature_cols].values.astype(np.float32)
        self.y = df[target_col].values.astype(np.int64)

        self.timesteps = FEATURE_COL_COUNT
        self.feature_dim = 1

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        x = self.X[idx].reshape(self.timesteps, self.feature_dim)
        y = self.y[idx]
        return torch.tensor(x), torch.tensor(y)

# CLIENT SPLIT
def create_clients(dataset, num_clients=5):
    indices = np.arange(len(dataset))
    np.random.shuffle(indices)
    splits = np.array_split(indices, num_clients)

    clients = []
    for s in splits:
        clients.append(Subset(dataset, s))
    return clients

# ENCODER (3 CNN + 1 BiLSTM)
class RNNEncoder(nn.Module):
    def __init__(self, input_dim=1, projection_dim=64):
        super().__init__()

        # CNN layers
        self.conv1 = nn.Conv1d(
            in_channels=1,
            out_channels=128,
            kernel_size=3,
            padding=1
        )

        self.conv2 = nn.Conv1d(
            in_channels=128,
            out_channels=64,
            kernel_size=3,
            padding=1
        )

        self.conv3 = nn.Conv1d(
            in_channels=64,
            out_channels=32,
            kernel_size=3,
            padding=1
        )

        # BiLSTM
        self.bilstm = nn.LSTM(
            input_size=32,
            hidden_size=64,
            batch_first=True,
            bidirectional=True
        )

        # Projection layer
        self.fc = nn.Linear(128, projection_dim)  # 64*2

    def forward(self, x):

        # (B, T, 1) -> (B, 1, T)
        x = x.permute(0, 2, 1)

        # CNN layers
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))

        # (B, C, T) -> (B, T, C)
        x = x.permute(0, 2, 1)

        # BiLSTM
        out, _ = self.bilstm(x)

        # Last timestep
        out = out[:, -1, :]

        # Projection
        z = self.fc(out)

        return F.normalize(z, dim=1)

# SUPCON LOSS
class SupConLoss(nn.Module):
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature

    def forward(self, features, labels):
        device = features.device
        labels = labels.view(-1, 1)

        mask = torch.eq(labels, labels.T).float().to(device)

        logits = torch.matmul(features, features.T) / self.temperature
        logits_mask = torch.ones_like(mask) - torch.eye(features.size(0), device=device)
        mask = mask * logits_mask

        exp_logits = torch.exp(logits) * logits_mask
        log_prob = logits - torch.log(exp_logits.sum(1, keepdim=True) + 1e-12)

        denom = torch.clamp(mask.sum(1), min=1.0)
        loss = -(mask * log_prob).sum(1) / denom

        return loss.mean()

# LOCAL TRAINING (MERGED LOSS)
def local_train(model, loader):
    model.train()

    con_loss_fn = SupConLoss().to(DEVICE)
    cls_loss_fn = nn.BCELoss()

    # Temporary classifier (local only)
    temp_classifier = nn.Sequential(
        nn.Linear(64, 1),
        nn.Sigmoid()
    ).to(DEVICE)

    optimizer = torch.optim.Adam(
        list(model.parameters()) + list(temp_classifier.parameters()), lr=LR
    )

    for _ in range(LOCAL_EPOCHS):
        for x, y in loader:
            x = x.to(DEVICE)
            y = y.float().to(DEVICE)

            optimizer.zero_grad()

            z = model(x)
            preds = temp_classifier(z).squeeze(1)

            loss_con = con_loss_fn(z, y.long())
            loss_cls = cls_loss_fn(preds, y)

            loss = (1 - LAMBDA) *loss_cls  + LAMBDA * loss_con

            loss.backward()
            optimizer.step()

    return model.state_dict()

# TRIMMED MEAN AGGREGATION
def aggregate(global_model, client_weights, trim_ratio=0.2):
    new_weights = {}

    for key in global_model.state_dict().keys():

        # Stack client parameters
        stacked_weights = torch.stack(
            [client_weights[i][key].float() for i in range(len(client_weights))]
        )

        # Sort weights across clients
        sorted_weights, _ = torch.sort(stacked_weights, dim=0)

        # Number of clients to trim
        k = int(trim_ratio * len(client_weights))

        # Avoid removing everything
        if 2 * k >= len(client_weights):
            trimmed_weights = sorted_weights
        else:
            trimmed_weights = sorted_weights[k: len(client_weights) - k]

        # Mean after trimming
        new_weights[key] = trimmed_weights.mean(dim=0)

    global_model.load_state_dict(new_weights)

    return global_model

# FEDERATED TRAINING
def federated_training(global_model, clients):
    global_model.to(DEVICE)

    for r in range(ROUNDS):
        client_weights = []

        for client_data in clients:
            loader = DataLoader(client_data, batch_size=BATCH_SIZE, shuffle=True)

            local_model = RNNEncoder().to(DEVICE)
            local_model.load_state_dict(global_model.state_dict())

            weights = local_train(local_model, loader)
            client_weights.append(weights)

        global_model = aggregate(global_model, client_weights)
        print(f"Round {r+1}/{ROUNDS} completed")

    return global_model

# CLASSIFIER (FINAL STAGE - UNCHANGED)
class Classifier(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
        self.fc = nn.Sequential(
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        with torch.no_grad():
            z = self.encoder(x)
        return self.fc(z).squeeze(1)

# CLASSIFIER TRAINING (UNCHANGED)
def train_classifier(model, loader):
    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    criterion = nn.BCELoss()

    for epoch in range(425):
        total_loss = 0
        for x, y in loader:
            x, y = x.to(DEVICE), y.float().to(DEVICE)

            optimizer.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"[Classifier] Epoch {epoch+1} Loss: {total_loss:.4f}")

    return model

# EVALUATION
def evaluate(model, loader):
    model.eval()
    preds, trues = [], []

    with torch.no_grad():
        for x, y in loader:
            x = x.to(DEVICE)
            out = model(x).cpu().numpy()

            preds.extend(out.tolist())
            trues.extend(y.numpy().tolist())

    preds_bin = [1 if p >= 0.5 else 0 for p in preds]

    # Confusion Matrix
    tn, fp, fn, tp = confusion_matrix(trues, preds_bin).ravel()

    # Specificity
    specificity = tn / (tn + fp)

    print("\nAccuracy:", accuracy_score(trues, preds_bin))
    print("Recall (Sensitivity):", recall_score(trues, preds_bin))
    print("Specificity:", specificity)
    print("Precision:", precision_score(trues, preds_bin))
    print("F1:", f1_score(trues, preds_bin))
    print("MCC:", matthews_corrcoef(trues, preds_bin))
    print("Kappa:", cohen_kappa_score(trues, preds_bin))
    print("AUC:", roc_auc_score(trues, preds))

# MAIN
def main():
    contrastive_dataset = FeatureDataset(CONTRASTIVE_CSV)
    clients = create_clients(contrastive_dataset, NUM_CLIENTS)

    global_model = RNNEncoder()

    # Federated training
    global_model = federated_training(global_model, clients)

    # Classifier training (UNCHANGED)
    train_ds = FeatureDataset(TRAIN_CSV)
    test_ds = FeatureDataset(TEST_CSV)

    train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True)
    test_loader = DataLoader(test_ds, BATCH_SIZE, shuffle=False)

    classifier = Classifier(global_model).to(DEVICE)
    classifier = train_classifier(classifier, train_loader)

    print("\nEvaluating...")
    evaluate(classifier, test_loader)

if __name__ == "__main__":
    main()

Round 1/25 completed
Round 2/25 completed
Round 3/25 completed
Round 4/25 completed
Round 5/25 completed
Round 6/25 completed
Round 7/25 completed
Round 8/25 completed
Round 9/25 completed
Round 10/25 completed
Round 11/25 completed
Round 12/25 completed
Round 13/25 completed
Round 14/25 completed
Round 15/25 completed
Round 16/25 completed
Round 17/25 completed
Round 18/25 completed
Round 19/25 completed
Round 20/25 completed
Round 21/25 completed
Round 22/25 completed
Round 23/25 completed
Round 24/25 completed
Round 25/25 completed
[Classifier] Epoch 1 Loss: 3.4117
[Classifier] Epoch 2 Loss: 3.3013
[Classifier] Epoch 3 Loss: 3.1943
[Classifier] Epoch 4 Loss: 3.0687
[Classifier] Epoch 5 Loss: 2.9222
[Classifier] Epoch 6 Loss: 2.7559
[Classifier] Epoch 7 Loss: 2.5783
[Classifier] Epoch 8 Loss: 2.3854
[Classifier] Epoch 9 Loss: 2.1870
[Classifier] Epoch 10 Loss: 2.0008
[Classifier] Epoch 11 Loss: 1.8105
[Classifier] Epoch 12 Loss: 1.6352
[Classifier] Epoch 13 Loss: 1.4899
[Classifier] 